# 09_windowing — effective window size

Supervisor request (meeting 2026-07-29): report the windowing measurement with
**mean and standard deviation**, compared against Diesner's window sizes.

Measured over **every** actor-concept link, including those in sentences
coreference rewrote. The measurement runs on the coref-resolved text, which is
the text 04_extract extracted the actor mentions from; lemmas are recomputed on
that text for the rewritten sentences so the lemma-matched concept triggers stay
locatable.

Reads `sentences_coref.jsonl` and `sentences_tagged.jsonl`. Writes to
`analysis/windowing/`. Touches no other notebook's output.

## Step 1: Setup, definition and Diesner benchmark

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import spacy

_cwd = Path().resolve()
ROOT = next((p for p in [_cwd] + list(_cwd.parents) if (p / 'src').is_dir()), _cwd)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

INTERIM_DIR = ROOT / 'data' / 'interim' / 'iran'
OUTPUT_DIR  = ROOT / 'data' / 'output' / 'iran'
DIR_WIN     = OUTPUT_DIR / 'analysis' / 'windowing'
DIR_WIN.mkdir(parents=True, exist_ok=True)

COREF_FILE  = INTERIM_DIR / 'sentences_coref.jsonl'
TAGGED_FILE = INTERIM_DIR / 'sentences_tagged.jsonl'

# ---------------------------------------------------------------------------
# WHAT THIS MEASURES
#
# Diesner (2012, p. 63) defines window size as "the number of space separated
# tokens that occur between the heads of the nodes in each annotated relation.
# The nodes themselves are not within the window", so two adjacent nodes give a
# window of zero. She disregards genitive markers, hyphens and single-character
# punctuation from the count. This notebook uses her definition, so the two are
# directly comparable.
#
# WHICH TEXT, and why it matters. The two endpoints of an edge are not derived
# from the same string by 04_extract: actors come from NER re-run on the
# COREF-RESOLVED text, while concepts are matched against the ORIGINAL text and
# its lemmas. Measuring on the original therefore loses every link whose actor
# mention coreference created, and those are not a random subset. This notebook
# measures on the resolved text for all sentences, recomputing lemmas for the
# rewritten ones so the lemma-matched concept triggers stay locatable. Every
# link is measured; none is dropped for being in a rewritten sentence.
# ---------------------------------------------------------------------------
PUNCT_SKIP = set(".,;:!?'\u2019\u2018\u201c\u201d\"()[]{}-\u2013\u2014/")

# Diesner's SemEval benchmark, unweighted average across relation types
# (2012, Table 37, p. 64): cumulative share of links found at each window size.
DIESNER_SEMEVAL = {0: 5.8, 1: 15.1, 2: 38.6, 3: 57.7, 4: 72.0, 5: 82.3,
                   6: 88.9, 7: 93.0, 8: 95.3, 9: 96.8, 10: 97.8, 11: 98.6,
                   12: 99.0}

print(f'Coref file  : {COREF_FILE}')
print(f'Tagged file : {TAGGED_FILE}')
assert COREF_FILE.exists(),  'ERROR: run 03b_coref first'
assert TAGGED_FILE.exists(), 'ERROR: run 04_extract first'


## Step 2: Linking sentences, resolved text and lemmas

In [ ]:
# The sentences that actually form links are the only ones worth measuring.
linking = set()
with open(TAGGED_FILE) as fh:
    for line in fh:
        d = json.loads(line)
        if d['actors'] and d['concepts']:
            linking.add(d['sentence_id'])

resolved, lemmas_by_id, orig_by_id, rewritten = {}, {}, {}, set()
n_sent = 0
with open(COREF_FILE) as fh:
    for line in fh:
        d = json.loads(line)
        n_sent += 1
        sid = d['sentence_id']
        rt = d.get('coref_resolved_text', d['text'])
        if rt != d['text']:
            rewritten.add(sid)
        if sid in linking:
            resolved[sid] = rt
            orig_by_id[sid] = d['text']
            lemmas_by_id[sid] = [str(x).lower() for x in d['lemmas']]

need_lemmas = [s for s in linking if s in rewritten]
print(f'Sentences                    : {n_sent}')
print(f'  rewritten by coreference   : {len(rewritten)}  ({len(rewritten)/n_sent:.1%})')
print(f'Linking sentences            : {len(linking)}')
print(f'  of those, rewritten        : {len(need_lemmas)}  '
      f'({len(need_lemmas)/len(linking):.1%})')

# Lemmas stored by 03_preprocess belong to the ORIGINAL text. For the rewritten
# linking sentences they must be recomputed on the resolved text, or the
# lemma-matched concept triggers cannot be located. Same model as 03, so the
# lemmas are the ones the pipeline would have produced.
nlp = spacy.load('en_core_web_trf')
if need_lemmas:
    texts = [resolved[s] for s in need_lemmas]
    for sid, doc in zip(need_lemmas, nlp.pipe(texts, batch_size=32)):
        lemmas_by_id[sid] = [t.lemma_.lower() for t in doc if not t.is_space]
    print(f'Lemmas recomputed on the resolved text for {len(need_lemmas)} sentences')


## Step 3: Measure every link

In [ ]:
def token_spans(text):
    """Tokens as (lower_text, char_start, char_end), whitespace dropped."""
    return [(t.text.lower(), t.idx, t.idx + len(t.text))
            for t in nlp.tokenizer(text) if not t.is_space]


def char_span_to_tokens(toks, lo, hi):
    return [i for i, (_, a, b) in enumerate(toks) if a < hi and b > lo]


def find_surface(toks, text_low, surface):
    out, start, s = [], 0, surface.lower().strip()
    if not s:
        return out
    while True:
        k = text_low.find(s, start)
        if k < 0:
            break
        idx = char_span_to_tokens(toks, k, k + len(s))
        if idx:
            out.append((idx[0], idx[-1]))
        start = k + max(1, len(s))
    return out


def gap(toks, a, b):
    """Diesner's window size: tokens strictly between the two node extents,
    genitives, hyphens and single-character punctuation disregarded."""
    (a0, a1), (b0, b1) = sorted([a, b])
    if b0 <= a1:
        return 0
    between = [toks[i][0] for i in range(a1 + 1, b0)]
    return sum(1 for w in between
               if not (len(w) == 1 and w in PUNCT_SKIP) and w not in ("'s", "\u2019s"))


rows = []
n_measured = n_align_fail = n_unlocatable = 0
with open(TAGGED_FILE) as fh:
    for line in fh:
        d = json.loads(line)
        sid = d['sentence_id']
        if sid not in linking:
            continue
        text = resolved[sid]
        toks = token_spans(text)
        lem = lemmas_by_id[sid]
        if len(toks) != len(lem):
            n_align_fail += 1
            continue
        text_low = text.lower()
        orig_low = orig_by_id[sid].lower()

        # An actor surface absent from the ORIGINAL text was put there by
        # coreference. Flagging it lets the two kinds be compared below.
        actor_pos, actor_coref = {}, {}
        for e in d['alias_log']:
            spans = find_surface(toks, text_low, e['surface'])
            if not spans:
                continue
            actor_pos.setdefault(e['resolved'], []).extend(spans)
            if e['surface'].lower().strip() not in orig_low:
                actor_coref[e['resolved']] = True

        concept_pos = {}
        for e in d['concept_log']:
            trig = str(e['trigger']).lower()
            spans = ([(i, i) for i, l in enumerate(lem) if l == trig]
                     if e['match_type'] == 'lemma'
                     else find_surface(toks, text_low, trig))
            if spans:
                concept_pos.setdefault(e['concept'], []).extend(spans)

        n_measured += 1
        for actor, aspans in actor_pos.items():
            for concept, cspans in concept_pos.items():
                dists = [gap(toks, a, c) for a in aspans for c in cspans]
                if not dists:
                    n_unlocatable += 1
                    continue
                rows.append({'window': d['window'], 'sentence_id': sid,
                             'actor': actor, 'concept': concept,
                             'distance': min(dists),
                             'actor_from_coref': bool(actor_coref.get(actor, False)),
                             'sentence_rewritten': sid in rewritten})

dist_df = pd.DataFrame(rows)
dist_df.to_csv(DIR_WIN / 'window_distances.csv', index=False)
print(f'Linking sentences measured   : {n_measured} of {len(linking)}')
print(f'  token/lemma drift skipped  : {n_align_fail}')
print(f'Links measured               : {len(dist_df)}')
print(f'  endpoints not located      : {n_unlocatable}')
print('\nSaved: window_distances.csv')


## Step 4: Summary, coverage and mention origin

In [ ]:
x = dist_df.distance.to_numpy()
cum = {k: float((x <= k).mean() * 100) for k in range(0, 26)}

summary = pd.DataFrame([{
    'n_links': len(x), 'mean': x.mean(), 'sd': x.std(ddof=1),
    'median': float(np.median(x)), 'p75': float(np.percentile(x, 75)),
    'p80': float(np.percentile(x, 80)), 'p90': float(np.percentile(x, 90)),
    'p95': float(np.percentile(x, 95)), 'max': int(x.max()),
    'within_2': cum[2], 'within_7': cum[7], 'within_12': cum[12],
}])
summary.to_csv(DIR_WIN / 'windowing_summary.csv', index=False)

cov = pd.DataFrame({'window_size': list(range(16)),
                    'ours_cumulative_pct': [cum[k] for k in range(16)],
                    'diesner_semeval_pct': [DIESNER_SEMEVAL.get(k, np.nan)
                                            for k in range(16)]})
cov.to_csv(DIR_WIN / 'windowing_coverage.csv', index=False)

# Coverage is now total, so the only thing left to report is how the two kinds
# of actor mention compare: the ones the text stated outright, and the ones
# coreference put there in place of a pronoun.
grp = (dist_df.groupby('actor_from_coref').distance
       .agg(['count', 'mean', 'std', 'median']).round(3))
grp.index = ['stated in the sentence', 'supplied by coreference']
grp.to_csv(DIR_WIN / 'windowing_by_mention_origin.csv')

by_win = (dist_df.groupby('window').distance
          .agg(['count', 'mean', 'std', 'median']).round(3))
by_win.to_csv(DIR_WIN / 'windowing_by_window.csv')

print('Saved: windowing_summary.csv, windowing_coverage.csv, '
      'windowing_by_mention_origin.csv, windowing_by_window.csv\n')
print(summary.round(3).T.to_string())
print()
print(cov.round(1).to_string(index=False))
print()
print(grp.to_string())
print()
print(by_win.to_string())
pct80 = int(np.ceil(np.percentile(x, 80)))
print(f'\n80 % of actor-concept links sit within {pct80} words of each other.')
print(f'Mean {x.mean():.2f}, SD {x.std(ddof=1):.2f}, median {np.median(x):.0f}.')


## Step 5: Figure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
TEAL, PURPLE, GREY, SIENNA = '#1f8fa6', '#7e57c2', '#999999', '#c0392b'

ax = axes[0]
cap = 25
ax.hist(np.clip(x, 0, cap), bins=np.arange(0, cap + 2) - .5, color=TEAL,
        edgecolor='white', linewidth=.6)
ax.axvline(x.mean(), color=PURPLE, lw=2.2, ls='--',
           label=f'mean {x.mean():.2f}, SD {x.std(ddof=1):.2f}')
ax.axvline(np.median(x), color=SIENNA, lw=2, ls=':',
           label=f'median {np.median(x):.0f}')
ax.set_xlabel('words between actor and concept trigger')
ax.set_ylabel('links')
ax.set_title(f'(a) All {len(x):,} links\n(last bin holds everything at {cap}+)',
             fontsize=10)
ax.legend(fontsize=8); ax.grid(axis='y', alpha=.3)

ax = axes[1]
ks = list(range(16))
ax.plot(ks, [cum[k] for k in ks], marker='o', lw=2.2, color=TEAL,
        label='this corpus')
dk = [k for k in ks if k in DIESNER_SEMEVAL]
ax.plot(dk, [DIESNER_SEMEVAL[k] for k in dk], marker='s', lw=2.2, ls='--',
        color=GREY, label='Diesner 2012, SemEval avg.')
for k in (2, 7, 12):
    ax.axvline(k, color='#dddddd', lw=1, zorder=0)
    ax.text(k, 2, f'{k}', ha='center', fontsize=8, color='#888888')
ax.set_xlabel('window size (words between the two nodes)')
ax.set_ylabel('cumulative % of links covered')
ax.set_ylim(0, 103)
ax.set_title("(b) Coverage against Diesner's benchmark", fontsize=10)
ax.legend(fontsize=8, loc='lower right'); ax.grid(alpha=.3)

# Panel (c) removed by request. The mention-origin comparison it drew is
# still written to windowing_by_mention_origin.csv and quoted in Methods.
fig.suptitle('Effective window: how far apart the linked terms actually sit',
             fontsize=12)
plt.tight_layout()
out = DIR_WIN / 'windowing.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved: {out.name}')


## Step 6: Validation checkpoint

In [ ]:
cov_links = len(dist_df)
print('VALIDATION CHECKPOINT (09_windowing):')
print('  Definition        : Diesner (2012, p. 63), tokens strictly between the')
print('                      node extents, punctuation and genitives disregarded')
print('  Measured on       : the coref-resolved text, which is the text the')
print('                      actor mentions were extracted from')
print(f'  Linking sentences : {n_measured} measured of {len(linking)}'
      f'   ({n_measured/len(linking):.1%})')
print(f'  Links measured    : {cov_links}')
print(f'  Drift skipped     : {n_align_fail} sentences')
print(f'  Not located       : {n_unlocatable} links')
print()
print(f'  mean {x.mean():.2f}   SD {x.std(ddof=1):.2f}   median {np.median(x):.0f}'
      f'   p80 {np.percentile(x, 80):.0f}   p95 {np.percentile(x, 95):.0f}')
print(f'  within  2 words: {cum[2]:.1f} %   (Diesner {DIESNER_SEMEVAL[2]} %)')
print(f'  within  7 words: {cum[7]:.1f} %   (Diesner {DIESNER_SEMEVAL[7]} %)')
print(f'  within 12 words: {cum[12]:.1f} %   (Diesner {DIESNER_SEMEVAL[12]} %)')
print()
for f in ('window_distances.csv', 'windowing_summary.csv',
          'windowing_coverage.csv', 'windowing_by_mention_origin.csv',
          'windowing_by_window.csv', 'windowing.png'):
    print(f'    [{"OK" if (DIR_WIN / f).exists() else "MISS"}]  analysis/windowing/{f}')
print()
print('This is a CHARACTERISATION of the design, not a defence of it. It says')
print('how far apart the linked terms actually sit; it does not claim that a')
print('sentence-level window recovers asserted semantic relations.')
